# Learning Notes

Personal log of what I've learned working on this project: concepts, tools, gotchas, things I want to remember.

## What is YAML

YAML ("YAML Ain't Markup Language") is a human-readable format for structured data, mostly used for config files. It uses indentation instead of brackets/braces, `key: value` pairs, and `- ` for list items.

The `data.yaml` file the Roboflow dataset download produced is a real example:

```yaml
path: C:/Users/rohan/Desktop/Quant Sports Project/experiments/Football-Players-Detection-1
train: train/images
val: valid/images
test: test/images

nc: 4
names: ['ball', 'goalkeeper', 'player', 'referee']
```

- `path`, `train`, `val`, `test`: tell YOLO where to find the dataset's image folders.
- `nc`: number of classes (4 here).
- `names`: the class labels, in the same order the model will predict them.

Nesting works by indentation, for example the `roboflow:` block underneath that has its own `workspace`, `project`, `version`, etc. as sub-keys. No special syntax needed, just consistent indentation.


## COCO weights, and why fine-tuning beats starting from scratch

COCO (Common Objects in Context) is a large general-purpose dataset, about 330,000 images across 80 everyday object classes (person, car, dog, sports ball, tv, etc.). "COCO-pretrained weights" means the model has already been trained on that dataset, so it already knows general visual concepts: edges, shapes, textures, what a person-shaped blob looks like from different angles, and so on.

That's exactly what `yolo26n.pt` was in the first smoke test: a COCO-pretrained checkpoint, used purely for inference (no training). It worked reasonably for the generic `person` class but was unreliable for `sports ball` and had no concept of soccer-specific roles (player vs. referee vs. goalkeeper), because COCO never taught it those distinctions.

**Fine-tuning** (what the Roboflow retrain decision is doing) means continuing training from those COCO-pretrained weights on a new, task-specific dataset, instead of starting from random weights. This is a form of transfer learning:

- The early layers of the network already know general visual features, so training doesn't have to relearn "what an edge looks like" from zero.
- Only the later layers really need to adapt, mainly the classification head, to the new classes (`ball`, `goalkeeper`, `player`, `referee`).
- This means far less data and far less compute is needed to get a good result, compared to training an object detector completely from scratch, which typically needs hundreds of thousands of images.

This is why the plan is "fine-tune YOLO26 on the Roboflow dataset" rather than "train a brand new detector from nothing."


## Why 4GB VRAM is little, and what CUDA actually does

**VRAM** is the memory that lives on the GPU itself (separate from regular system RAM). During training, VRAM has to hold: the model's weights, the batch of images currently being processed, all the intermediate activations from the forward pass, and the gradients from the backward pass, all at once. All of that competes for the same pool of memory.

4GB is small by current standards. The GTX 1650 in this machine is a budget/laptop-class GPU; modern training GPUs (RTX 4090, A100, H100) have 24GB to 80GB+. The practical consequence: if the batch size or image size is too large for the available VRAM, training crashes with an out-of-memory (OOM) error rather than just running slower. That's why the training run started with `batch=16` instead of YOLO's larger default, as a conservative starting point, with room to raise it if VRAM allows or lower it if it OOMs.

**CUDA** is NVIDIA's platform that lets software run computations directly on the GPU instead of the CPU. The reason this matters for deep learning: a CPU has a small number of powerful, general-purpose cores, while a GPU has thousands of simpler cores built to do the same operation on lots of data at once. Deep learning is mostly large matrix multiplications, which is exactly the kind of highly-parallel, repetitive math GPUs are built for. CUDA is the layer that lets PyTorch hand those matrix operations off to the GPU's cores instead of the CPU's.

Concretely, in this project: `torch` was first installed as a CPU-only build (`2.14.0+cpu`), so `torch.cuda.is_available()` returned `False` and training would have run on the CPU, slowly. Reinstalling `torch`/`torchvision` from the CUDA-enabled index (`cu130`, matching this machine's driver) fixed that; `torch.cuda.is_available()` now returns `True` and the GTX 1650 is detected and usable for training.


## Lesson (2026-09-18): local GPU training on old/underpowered hardware is a real hardware risk, not just slow

While attempting to run training locally on the GTX 1650, a capacitor on the machine blew, likely a short circuit related to the power adapter under sustained load. Fixed, no lasting damage, but the practical lesson is:

Training deep learning models pushes a machine's power delivery and cooling much harder than normal use, for a sustained period (not a quick spike). On older or already-marginal hardware, that sustained load is a genuine risk to the hardware itself, not just something that runs slowly.

**Decision going forward:** use an external/cloud GPU (Google Colab, or similar) for actual training runs instead of pushing local hardware. Local setup is still fine for quick inference smoke tests (like the original YOLO detection test), just not for sustained training workloads on this machine.


## Workflow (2026-09-20): shifting training from local to Google Colab

Following the local GPU hardware fault (previous entry), the actual fine-tuning run moves to Google Colab instead of this machine. Two ways to drive Colab, and why one was picked over the other:

**Option A: the official `google-colab-cli`.** Lets you provision a GPU (`colab new -s mysession --gpu T4`), install deps, and run a plain `.py` script unattended (`colab exec -f train.py`) entirely from a terminal, no browser tab needed. Built for headless/automated runs. Problem: it only supports Linux and macOS, not Windows, so on this machine it would need WSL2 first.

**Option B: upload a notebook to colab.research.google.com and run it interactively.** No install, no WSL, works from any browser. This is the one actually used, given the Windows constraint. Practical differences from running the same code locally:
- Colab's disk is remote and ephemeral (wiped when the session ends), so local file paths (like the smoke test's hardcoded `C:\Users\rohan\...\08fd33_4.mp4`) mean nothing there. Anything the training code reads either has to be re-downloaded inside Colab (the Roboflow dataset, via its API) or mounted from Google Drive.
- Secrets can't come from a local `.env` file, since that file never leaves this machine. Colab has its own per-notebook Secrets panel instead (see the next entry).
- Trained weights need to be pulled back out before the session ends: either download the file directly, or mount Drive and save there during training.

**Where the code lives:** the training notebook is `pipeline/detection/train.ipynb`, not `experiments/`. `experiments/` is reserved (per `decision_log.md`, 2026-09-18) for diagnostic spikes like the pretrained YOLO smoke test. This training run is the actual Stage 1 pipeline deliverable the smoke test's findings led to, not a spike, so it gets its own real location, one that mirrors CLAUDE.md's numbered pipeline stages (`pipeline/detection/` = Stage 1, `pipeline/calibration/` would be Stage 2, and so on).


## Lesson (2026-09-20): why `.env` (or any secret) must never be committed to git

`.env` holds the Roboflow API key as a plain `KEY=value` line, read in Python via `python-dotenv`'s `load_dotenv()` plus `os.environ["ROBOFLOW_API_KEY"]`. It's listed first in `.gitignore` for a reason: if a secret ever gets committed, deleting it in a later commit does **not** remove it. Git keeps full history, so the key still sits in the repo's `.git` history, retrievable by anyone with access to the repo, forever, unless the history itself is rewritten (a destructive operation, and not a reliable fix once something's been pushed anywhere public). On a public repo, bots actively scan new commits for exposed API keys within minutes, so a leaked key there should be treated as immediately compromised, not just "risky."

The same risk shows up in a different shape on Colab. There's no `.env` file there (see previous entry), but pasting the raw key directly into a notebook code cell has the identical problem: if that notebook is ever saved and pushed to the repo with the key still typed into a cell, the key is committed exactly as if it were in a tracked `.env` file. That's why the Colab version uses `google.colab.userdata.get("ROBOFLOW_API_KEY")` instead: the key lives in Colab's Secrets panel (tied to the Google account, not the notebook file), and `userdata.get(...)` only pulls the value in at runtime. Nothing about the actual key ever gets written into the `.ipynb` file's saved source.

General rule this reinforces: secrets flow through environment variables or a dedicated secret store, read at runtime, never typed as a literal value into any file that gets committed.


## Concept (2026-09-20): reading YOLO training output, loss terms and mAP

Every epoch, `yolo mode=train` prints a row with three loss values and, after each epoch, a validation row with precision, recall, and two mAP scores. They measure different things and get watched differently.

**The three losses (should trend down over epochs):**
- `box_loss`: how far off the predicted bounding box's position and size are from the ground-truth box (a CIoU-style loss: penalizes overlap, center distance, and aspect ratio mismatch together).
- `cls_loss`: how confidently and correctly the model predicts the right class for each detected box (`ball`, `goalkeeper`, `player`, `referee` in this dataset). High `cls_loss` means it's unsure or picking the wrong class, not that boxes are misplaced.
- `l1_loss`: a simple L1 penalty on the raw box coordinate error. It is tiny in magnitude (around 0.001 to 0.002 in the first run), so it barely moves compared to the other two.

**Correction (2026-09-20):** an earlier version of this entry listed `dfl_loss` (Distribution Focal Loss) as the third loss. That is what YOLOv8 and YOLO11 report, but YOLO26 removed DFL from its detection head and replaced it with direct box regression plus L1 loss, which is why the real training log shows `l1_loss` instead. Lesson: the loss columns depend on the model version, so check the actual log rather than assuming from older YOLO docs.

All three are *training-set* losses: what the optimizer is directly minimizing during the backward pass. Going down means the model fits the training data better; it does not by itself mean the model generalizes.

**`mAP50` and `mAP50-95` (should trend up, evaluated on the validation split after every epoch):**
- A predicted box counts as a correct detection if it overlaps the ground-truth box by at least some IoU (Intersection over Union) threshold. `mAP50` requires 50% overlap; `mAP50-95` averages accuracy across ten thresholds from 50% to 95% in 5% steps, a stricter, more comprehensive score (the same metric standard in the COCO benchmark).
- "Average Precision" itself is the area under a class's precision-recall curve; "mean" AP averages that across all classes (ball, goalkeeper, player, referee here), so one weak class quietly pulls the whole number down even if the others are strong.

**Why watch both, not just the losses:** loss is computed on training data and always has some incentive to keep dropping; mAP is computed on held-out validation data and is the actual measure of whether the model works on data it wasn't trained on. If losses keep falling while `mAP50-95` plateaus or drops, that's the classic overfitting signal, worth stopping the run for rather than letting all 100 epochs finish. A `nan` appearing in any loss column is a different failure mode entirely (usually a learning-rate or bad-data problem), and also worth stopping for immediately rather than waiting it out.


## Concept (2026-09-20): the `runs/` folder, and why it must be gitignored

Every time Ultralytics runs a `train`, `val`, or `predict` command, it automatically creates a `runs/` folder next to wherever the command was run from, and saves that run's output in a numbered subfolder (for example `runs/detect/train/`, then `train2/`, `train3/` on later runs). Nothing has to be configured for this, which is why a `runs/` folder appeared without anyone creating it.

**What ends up inside:**
- `weights/best.pt` and `weights/last.pt`: the trained checkpoints.
- `results.csv` and `results.png`: the per-epoch losses and mAP, as numbers and as curves.
- Confusion matrix and precision-recall plots.
- Annotated images: sample training batches and validation predictions with boxes drawn on them. For `predict` runs on video, the output is the annotated video itself.

**Why it needs a `.gitignore` entry:**
- **Derived frames.** The annotated images and videos are derived from footage. CLAUDE.md's rule is that public repo output may contain code and aggregated statistics only, never raw video or derived frames, because of DFL and SoccerNet redistribution restrictions. Committing `runs/` by accident would break that rule.
- **Size and reproducibility.** The contents are large and can be regenerated by re-running the command, so they add weight to the repo without adding anything that needs version history.

**How the pattern works:** the `.gitignore` already had `experiments/runs/`, but that only matches that one exact path. A run from `pipeline/detection/` creates `pipeline/detection/runs/`, which the old line would miss. A pattern with no folder prefix, just `runs/`, matches a folder with that name at any depth in the repo. The trailing slash means it only matches directories.

**Gotcha:** `.gitignore` only affects files git is not already tracking. If a file was committed before its ignore rule existed, it stays tracked, and it has to be removed from git explicitly (`git rm --cached`). Adding the rule early, before any run output exists, is the safe order. To check that a path is ignored, run `git check-ignore -v <path>`, which prints the rule that matched.


## Concept (2026-09-21): `YOLO("file.pt")` is how you pick which weights a model uses

```python
from ultralytics import YOLO

model = YOLO("football_yolo26n_best.pt")
results = model.predict("clip.mp4", save=True)
```

The string passed to `YOLO(...)` is the path to a weights file (a `.pt` checkpoint). That one argument decides which trained model you get. Nothing else in the code changes:
- `YOLO("yolo26n.pt")` loads the stock COCO model (80 classes: person, sports ball, tv, ...).
- `YOLO("football_yolo26n_best.pt")` loads the model fine-tuned on Colab (4 classes: ball, goalkeeper, player, referee).

**This is how off-site training gets used locally.** Training ran on Colab's GPU and produced `best.pt` there. Downloading that file and passing its path to `YOLO(...)` on this machine is the whole hand-off. The `.pt` file bundles the network architecture, the learned weights, and the class names, so the local code does not need to know anything about how or where it was trained. Loading it is the same step whether the weights came from Colab, from a local run, or from Ultralytics' pretrained downloads.

**Checking which model you actually loaded:** `model.names` lists the classes it can predict. The COCO model has 80 entries and the fine-tuned one has 4, which is how the two `.pt` files in `pipeline/detection/` were told apart.

**Path gotcha:** a relative path like `"football_yolo26n_best.pt"` is resolved from the notebook's own folder (`pipeline/detection/`), not the project root. If the path is wrong, the notebook fails with a file-not-found error. The exception is official Ultralytics names such as `yolo26n.pt`: if that file is missing, Ultralytics silently downloads it into the current folder, which is why a stray copy of the COCO checkpoint appeared in `pipeline/detection/`.

The command-line version of the same thing is the `model=` argument, for example `yolo task=detect mode=val model=football_yolo26n_best.pt ...`.


## Concept (2026-09-21): what OpenCV (`cv2`) is, and where it sits in the pipeline

`cv2` is OpenCV, a general-purpose library for images and video. It is installed as `opencv-python` but imported as `cv2` (a naming leftover from the library's history). Ultralytics already depends on it, which is why it was in the `.venv` without being installed separately.

**Split to remember:** YOLO (Ultralytics, running on PyTorch) decides *what is where* in a frame. OpenCV handles *pixels, video, and geometry*.

**Where cv2 shows up in the pipeline:**
- **Before detection:** `cv2.VideoCapture` reads frames, `cv2.VideoWriter` saves them.
- **After detection, presentation:** drawing rings, triangles, and labels (`cv2.ellipse`, `cv2.drawContours`, `cv2.putText`).
- **After detection, data:** this is not presentation. `cv2.findHomography` and `cv2.warpPerspective` convert pixel positions into real pitch coordinates (Stage 2, calibration), and every speed or distance figure depends on it. `cv2.kmeans` on jersey pixels can assign teams (Stage 3). These steps produce numbers, not pictures.
- **Tracking itself** (ByteTrack and similar) mostly works from the detection boxes with motion math, not from OpenCV.

Pipeline order: `read frames (cv2) -> detect (YOLO) -> track (tracker) -> calibrate and assign teams (cv2) -> features -> draw (cv2, optional)`.

For live trading, drawing is the only optional step. It is useful for debugging, but trading logic runs on coordinates and features. Calibration is not optional.


## Concept (2026-09-21): drawing on frames with cv2, and why it must work one frame at a time

Code lives in `pipeline/common/drawing.py` (`draw_ellipse`, `draw_triangle`, `annotate_frame`). Shared helpers go in `pipeline/common/`, not in one stage's folder.

**Coordinates.** A frame is a NumPy array of shape `(height, width, 3)`. The origin `(0, 0)` is the top-left, x grows right, and y grows *down*. So the bottom of a box is its larger y (`y2`), and "above the ball" means a smaller y.

**Positions versus sizes.** `x1, y1, x2, y2` are positions (where something is). `width = x2 - x1` is a size (how big it is). A size comes from a difference, never from a raw position. Setting the ring's half-height to `y2 / 2` was a mistake: `y2` depends on where the player stands in the frame, so the ring would change size with position. Half-height should come from the box width instead (`b = max(1, int(0.35 * width))`).

**`cv2.ellipse(frame, center, axes, angle, startAngle, endAngle, color, thickness)`.**
- `center` is `(cx, y2)`, the bottom-center of the box, so the ring sits at the feet.
- `axes` is `(a, b)`, the half-width and half-height. The ring is squashed (`b` smaller than `a`) because the camera views the pitch at an angle.
- The start and end angles are in degrees, measured clockwise from the right side (3 o'clock), because y points down. `0` to `360` is a full ring. `-45` to `235` leaves a gap at the top.

**Gotchas hit while writing it:**
- **Colors are BGR, not RGB.** `(0, 255, 255)` is yellow. Showing a frame with matplotlib needs `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` first, or the colors look swapped.
- **`/` gives a float, `//` gives an int.** `cx = (x1 + x2) / 2` crashed because OpenCV only accepts integer coordinates. The error message was misleading (`ellipse() takes at most 5 arguments (8 given)`); it really meant the center could not be parsed. Use `//`.
- **Return the frame.** A function ending in `pass` returns `None`, so `frame = draw_ellipse(...)` would replace the frame with `None`. OpenCV also draws directly on the array it is given, so the original frame changes; call `frame.copy()` first to keep an untouched version.

**Triangle.** Three `[x, y]` points in a NumPy array with dtype `np.int32`: the tip just above the ball's box, and two corners higher up to the left and right. `cv2.drawContours(frame, [points], 0, color, cv2.FILLED)` fills it; the points go inside a list and `0` means "the first contour". A second call with a black color and thickness 2 adds an outline so it stays visible on the pitch.

**Reading results.** `result.boxes.xyxy` is a torch tensor, so `.tolist()` gives plain numbers. `result.names[int(cls)]` turns a class number into its name (`ball`, `player`, ...). A `COLORS` dictionary with `.get(name, default)` avoids a crash on an unknown class.

**Why per frame matters (live).** The drawing functions take one frame in and return one frame out, so they work identically on a file or a live stream. The batch pattern does not: a `read_video` that returns a list of every frame needs about 6 MB per 1080p frame, roughly 4.4 GB for this 750-frame clip and hundreds of GB for a full match, and a live stream never ends. The fix is a generator (a function that uses `yield` to hand back one frame at a time), the same idea as Ultralytics' `stream=True`. Related traps in that code: a hardcoded 24 fps when the clip is 25 fps (speed and distance depend on real timing), no check that the file opened, and a missing `cap.release()`. For live, also budget time: at 25 fps there are 40 ms per frame, and detection alone took about 45 ms on CPU.

**Licensing note.** The reference repo (`abdullahtarek/football_analysis`) declares no license, so its code is all-rights-reserved by default. The drawing style (an ellipse at the feet, a triangle for the ball) was reimplemented rather than copied.

**Notebook import trick.** A notebook runs from its own folder, so to import from `pipeline/common/`, add the project root to the path (`sys.path.append("../..")`) and turn on `%load_ext autoreload` with `%autoreload 2` so edits to the `.py` file are picked up without restarting the kernel.
